# Exploring a recording

A starting point for working on the order book, the factors and the cost model
offline. Everything below runs against the committed sample; point `SOURCE` at a
real capture in `data/raw/` once you have one.

In [ ]:
from pathlib import Path

import polars as pl

from l2tca.feed.replay import JsonlReplay
from l2tca.feed.parser import BookFrame, parse_frame

SOURCE = Path("../data/samples/mock_btcusd_book_2min.jsonl.gz")
messages = list(JsonlReplay(SOURCE).iter_messages())
len(messages), messages[0]

## What is in the file

In [ ]:
frames = [(m, parse_frame(m.payload)) for m in messages]
pl.DataFrame(
    {"kind": [type(f).__name__ for _, f in frames], "session": [m.session for m, _ in frames]}
).group_by("kind").len().sort("len", descending=True)

## Message cadence

`recv_wall_ns` is the clock to use across sessions; `recv_ns` is only comparable
within one recording process.

In [ ]:
gaps_ms = pl.Series(
    "gap_ms",
    [
        (b.recv_wall_ns - a.recv_wall_ns) / 1e6
        for a, b in zip(messages, messages[1:])
        if a.session == b.session
    ],
)
gaps_ms.describe()

## Book frames, level by level

In [ ]:
book_frames = [(m, f) for m, f in frames if isinstance(f, BookFrame)]
snapshot_msg, snapshot = next((m, f) for m, f in book_frames if f.is_snapshot)
pl.DataFrame(
    {
        "side": ["bid"] * len(snapshot.bids) + ["ask"] * len(snapshot.asks),
        "price": [level.price for level in snapshot.bids + snapshot.asks],
        "qty": [level.qty for level in snapshot.bids + snapshot.asks],
    }
).head(10)

## Reading the Parquet tables

Run `uv run l2tca export data/samples/mock_btcusd_book_2min.jsonl.gz --out ../data/parquet`
first, then scan them lazily.

In [ ]:
from l2tca.io.reader import scan_table, validate_table

root = Path("../data/parquet")
if (root / "tick").exists():
    print(validate_table(root, "tick").describe())
    display(scan_table(root, "tick").head(5).collect())

## Once the order book is implemented

`OrderBook` is a specified stub; `tests/test_core_contract.py` is its acceptance
suite. When it applies frames, this cell replays the file through it and plots the
top of book.

In [ ]:
from l2tca.book import OrderBook

book = OrderBook(symbol="BTC/USD", depth=100)
mids = []
try:
    for m, f in book_frames:
        book.apply_snapshot(f) if f.is_snapshot else book.apply_update(f)
        top = book.top_of_book()
        mids.append({"recv_wall_ns": m.recv_wall_ns, "mid": top.mid, "spread": top.spread})
except NotImplementedError as exc:
    print(f"still a stub: {exc}")
else:
    display(pl.DataFrame(mids).tail())

## Latency of the hot path

Same harness the CLI runs, so a number measured here is comparable with
`uv run l2tca bench`.

In [ ]:
from l2tca.bench import run_suite

print(run_suite(messages, warmup=500).describe())